In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

customers = pd.read_csv("C:/Users/lenovo/Desktop/FinRisk-X/database/customers.csv")

customers.head()

,customer_id,age,annual_income,employment_years,dependents,savings,existing_debt,credit_score,employment_type,debt_to_income,savings_to_income,default_probability,default
0,100001,59,6.542970e+05,0,3,281262.935425,185332.880592,708,Contract,0.283255,0.429870,0.415757,1
1,100002,49,6.216075e+05,4,3,189110.449367,116700.181303,810,Salaried,0.187739,0.304228,0.296416,1
2,100003,35,6.371762e+05,10,2,396507.236148,67320.829790,595,Salaried,0.105655,0.622288,0.249349,1
3,100004,63,1.915457e+06,19,4,350710.898806,217865.937714,761,Salaried,0.113741,0.183095,0.216395,1
4,100005,28,3.833640e+05,10,4,324423.445016,446028.336260,689,Business Owner,1.163459,0.846254,0.592992,0


In [2]:
N_LOANS = 18000

loan_customer_ids = np.random.choice(
    customers["customer_id"],
    size=N_LOANS,
    replace=True
)

loans = pd.DataFrame({
    "loan_id": range(500001, 500001 + N_LOANS),
    "customer_id": loan_customer_ids
})

In [3]:
income_lookup = customers.set_index("customer_id")["annual_income"]

loans["annual_income"] = loans["customer_id"].map(income_lookup)

loans["loan_amount"] = (
    loans["annual_income"]
    * np.random.uniform(0.15, 1.2, N_LOANS)
)

loans["loan_amount"] = (
    loans["loan_amount"]
    .clip(50000, 5000000)
    .round(0)
)

In [4]:
loans["loan_term_months"] = np.random.choice(
    [12, 24, 36, 48, 60, 72],
    size=N_LOANS,
    p=[0.05, 0.10, 0.25, 0.20, 0.30, 0.10]
)

In [5]:
credit_lookup = customers.set_index("customer_id")["credit_score"]

loans["credit_score"] = loans["customer_id"].map(credit_lookup)

loans["interest_rate"] = (
    7
    + (750 - loans["credit_score"]) * 0.025
    + np.random.normal(0, 0.8, N_LOANS)
)

loans["interest_rate"] = loans["interest_rate"].clip(5, 25).round(2)

In [6]:
loan_purposes = [
    "Home",
    "Education",
    "Vehicle",
    "Personal",
    "Business",
    "Medical"
]

loans["loan_purpose"] = np.random.choice(
    loan_purposes,
    size=N_LOANS,
    p=[0.20, 0.10, 0.20, 0.25, 0.15, 0.10]
)

In [7]:
P = loans["loan_amount"]
r = loans["interest_rate"] / 100 / 12
n = loans["loan_term_months"]

loans["monthly_payment"] = (
    P * r * (1 + r) ** n
    / ((1 + r) ** n - 1)
)

loans["monthly_payment"] = loans["monthly_payment"].round(2)

In [8]:
loans.head()

,loan_id,customer_id,annual_income,loan_amount,loan_term_months,credit_score,interest_rate,loan_purpose,monthly_payment
0,500001,107271,4.759211e+05,167399.0,60,645,10.80,Home,3622.99
1,500002,100861,7.489607e+05,636946.0,24,722,8.21,Home,28868.39
2,500003,105391,1.012026e+06,762323.0,24,688,8.17,Personal,34536.94
3,500004,105192,5.800417e+05,279011.0,60,621,10.17,Vehicle,5951.52
4,500005,105735,5.152127e+05,354780.0,36,572,11.94,Home,11773.61


In [9]:
loans["loan_to_income"] = (
    loans["loan_amount"] / loans["annual_income"]
)

In [10]:
loans["monthly_income"] = loans["annual_income"] / 12

In [11]:
loans["payment_to_income"] = (
    loans["monthly_payment"] /
    loans["monthly_income"]
)

In [12]:
loan_risk = (
    0.35 * (1 - loans["credit_score"] / 850)
    + 0.30 * loans["loan_to_income"].clip(0, 1)
    + 0.20 * loans["payment_to_income"].clip(0, 1)
    + 0.15 * (loans["interest_rate"] / 25)
)

loan_risk = np.clip(loan_risk, 0, 1)

In [13]:
loans["default_probability"] = (
    1 / (1 + np.exp(-8 * (loan_risk - 0.45)))
)

In [14]:
loans["default"] = (
    np.random.random(N_LOANS)
    < loans["default_probability"]
).astype(int)

In [15]:
loans["default"].value_counts(normalize=True)

default
0    0.658111
1    0.341889
Name: proportion, dtype: float64

In [16]:
loans.to_csv(
    "C:/Users/lenovo/Desktop/FinRisk-X/database/loans.csv",
    index=False
)